# Phase 7: solver certification (independent scipy cross-check)

Re-derives the RS-385 motor transient with a THIRD integrator (`scipy.integrate.solve_ivp`, an adaptive RK45) and overlays it on the SpikyPanda simulation and the closed-form analytic transient. Then it refits the Cash-Karp convergence order from the fixed-step error table.

Run all cells from this folder. Needs `numpy`, `pandas`, `matplotlib`, `scipy`.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

tr = pd.read_csv('data/phase7-transient.csv')      # t, omega_sim, omega_ana, i_sim, i_ana
cv = pd.read_csv('data/phase7-convergence.csv')     # dt, error (fixed-step Cash-Karp)

# RS-385 motor parameters + the certification operating point (V, tau).
R, L, Ke, Kt, b, J = 1.22, 1e-3, 8.22e-3, 8.22e-3, 1.03e-6, 6e-7
V, tau = 7.0, 6e-3
print(f'{len(tr)} transient samples, {len(cv)} convergence points')
tr.head()

## 1. Three integrators agree: sim vs analytic vs scipy

In [ ]:
def motor_rhs(t, x):          # x = [current, omega]
    i, w = x
    return [(V - R * i - Ke * w) / L, (Kt * i - b * w - tau) / J]

sol = solve_ivp(motor_rhs, [0, tr.t.max()], [0.0, 0.0], t_eval=tr.t.values, rtol=1e-11, atol=1e-13)
omega_scipy = sol.y[1]

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(tr.t * 1e3, tr.omega_sim, color='tab:blue', lw=2.5, label='SpikyPanda sim')
ax.plot(tr.t * 1e3, tr.omega_ana, color='tab:red', lw=1.2, ls='--', label='analytic e^{At}')
ax.plot(tr.t * 1e3, omega_scipy, color='tab:green', lw=1.0, ls=':', label='scipy solve_ivp')
ax.set_xlabel('t [ms]'); ax.set_ylabel('speed omega [rad/s]')
ax.set_title('RS-385 motor speed from rest: three independent integrators coincide')
ax.legend(); ax.grid(alpha=0.3); fig.tight_layout(); plt.show()

## 2. Residuals: how far the sim is from exact theory

In [ ]:
scale = abs(tr.omega_ana.iloc[-1])   # steady speed, the relative-error scale
plt.figure(figsize=(9, 3.6))
plt.plot(tr.t * 1e3, (tr.omega_sim - tr.omega_ana) / scale * 100, color='tab:blue', label='sim - analytic')
plt.plot(tr.t * 1e3, (omega_scipy - tr.omega_ana) / scale * 100, color='tab:green', label='scipy - analytic')
plt.xlabel('t [ms]'); plt.ylabel('speed error [% of steady]')
plt.title('Both numerical integrators sit at their tolerance floor against the exact transient')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()
print('max |sim - analytic| / steady   =', float(abs(tr.omega_sim - tr.omega_ana).max() / scale))
print('max |scipy - analytic| / steady  =', float(abs(omega_scipy - tr.omega_ana).max() / scale))

## 3. Convergence order of the Cash-Karp RK4(5)

In [ ]:
slope, intercept = np.polyfit(np.log10(cv.dt), np.log10(cv.error), 1)
ref = 10 ** (intercept) * cv.dt ** slope
plt.figure(figsize=(9, 4))
plt.loglog(cv.dt, cv.error, 'o-', color='tab:purple', label=f'measured error (slope {slope:.2f})')
plt.loglog(cv.dt, ref, '--', color='gray', label='log-log fit')
# pure slope-5 guide through the coarsest point
guide = cv.error.iloc[0] * (cv.dt / cv.dt.iloc[0]) ** 5
plt.loglog(cv.dt, guide, ':', color='tab:red', label='slope 5 (theory)')
plt.xlabel('step dt [s]'); plt.ylabel('global error')
plt.title(f'Fixed-step Cash-Karp converges at order {slope:.2f} (theory 5)')
plt.legend(); plt.grid(alpha=0.3, which='both'); plt.tight_layout(); plt.show()
print('fitted order =', round(float(slope), 3))